# Lift rs-IMLE training on Colab GPU

A different task than the PushT experiments (see `colab/pusht_colab_train.ipynb`), to check
whether PushT's ~50-58% success-rate ceiling is specific to that task or a broader property of
this method/codebase. Same `train.py` entrypoint, same wandb account, same checkpoint/resume
mechanism -- only the task and its dependencies differ. See `colab/COLAB_WORKFLOW.md` for the
general workflow this follows.

**Runtime**: Runtime -> Change runtime type -> GPU. Lift uses `num_cameras=2` (vs PushT's 1)
and a 7-dim action space (vs PushT's 2), so per-step cost is higher than PushT even though the
default config's `batch_size=64` is lower than PushT's original 128 -- **the first training
cell below runs a short calibration (10 epochs) before committing to the full 500**, since we
don't have a measured per-epoch time for this task yet (see the estimate note in that cell).

## 1. Mount Google Drive
Same reasoning as the PushT notebook: Colab VMs are ephemeral, Drive persists across sessions
and disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/imle_policy_colab'
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/saved_weights', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/datasets', exist_ok=True)
print('Drive root:', DRIVE_ROOT)

## 2. Clone your fork

In [ ]:
GITHUB_USER = 'santhoshetty'  # your fork
!git clone https://github.com/{GITHUB_USER}/imle_policy.git /content/imle_policy
%cd /content/imle_policy
!git log --oneline -5

## 3. Install dependencies

Lift uses `robosuite` (the simulator/eval environment) and `mujoco`, not PushT's
`pymunk`/`pygame`. This is a heavier install than the PushT notebook's -- expect a few minutes,
and note `robosuite` is pinned to the exact version (`v1.4.1`) this repo's eval code was written
against (`pyproject.toml`), same principle as pinning `pymunk` in the PushT notebook.

In [ ]:
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q diffusers wandb 'mujoco<=3.1.6' 'imageio[ffmpeg]' opencv-python-headless \
    scikit-image numpy
!pip install -q "robosuite @ git+https://github.com/ARISE-Initiative/robosuite@v1.4.1"

import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

import robosuite
print('robosuite', robosuite.__version__)

## 4. Get the Lift dataset onto Drive (one-time)

Unlike PushT (which has its own small standalone zip), Lift's dataset is only distributed
bundled inside the project's full `datasets.zip` (**25.8GB, all tasks together** -- confirmed
via the HuggingFace API, there is no per-task zip for anything except PushT). Downloading and
fully extracting 25.8GB to Drive would eat most of a free Google account's Drive quota, so this
cell instead:
1. Downloads the big zip to the **Colab VM's local disk** (`/content`, not Drive -- ephemeral,
   but free, and Colab's default disk is generously sized for a temporary 25.8GB file).
2. Uses `zipfile` to find and extract **only** the Lift-related file(s) out of the archive
   (without fully unpacking the other tasks), copies that to Drive.
3. Deletes the local copy of the big zip.

After this runs once, `lift.pkl` lives permanently on Drive like `pusht.pkl` does, and every
future session just reuses it -- this cell becomes a no-op (skipped) once it's there.

**This will take a while the first time** -- 25.8GB at typical Colab download speeds is roughly
10-25 minutes depending on Google's network conditions that session; this is a one-time cost,
not a per-run cost.

In [ ]:
import os, zipfile, urllib.request

DRIVE_DATASET = f'{DRIVE_ROOT}/datasets/lift.pkl'
LOCAL_ZIP = '/content/datasets.zip'

if os.path.exists(DRIVE_DATASET):
    print('lift.pkl already on Drive, skipping download+extract:', DRIVE_DATASET)
else:
    if not os.path.exists(LOCAL_ZIP):
        print('Downloading datasets.zip (25.8GB, one-time)...')
        url = 'https://huggingface.co/datasets/krishanrana/imle_policy/resolve/main/datasets.zip'
        urllib.request.urlretrieve(url, LOCAL_ZIP)
        print('Download complete.')

    print('Scanning zip for Lift-related files...')
    with zipfile.ZipFile(LOCAL_ZIP) as zf:
        names = zf.namelist()
        # Don't assume the exact internal path -- search case-insensitively for anything
        # plausibly the Lift pickle rather than hardcoding a path we haven't verified.
        candidates = [n for n in names if 'lift' in n.lower() and n.lower().endswith('.pkl')]
        print('Found candidates:', candidates)
        assert candidates, (
            "No Lift-related .pkl found in datasets.zip -- run "
            "`zipfile.ZipFile('/content/datasets.zip').namelist()` yourself to inspect the "
            "actual structure and update the filter above."
        )
        # If more than one candidate matches, prefer the shortest path (least likely to be a
        # nested/alternate copy) and print the rest so you can sanity-check the choice.
        chosen = sorted(candidates, key=len)[0]
        print('Extracting:', chosen)
        zf.extract(chosen, '/content/extracted')

    import shutil
    shutil.move(f'/content/extracted/{chosen}', DRIVE_DATASET)
    print('Saved to Drive:', DRIVE_DATASET)

    os.remove(LOCAL_ZIP)
    shutil.rmtree('/content/extracted', ignore_errors=True)
    print('Cleaned up local zip/extraction.')

os.makedirs('/content/imle_policy/imle_policy/datasets', exist_ok=True)
local_link = '/content/imle_policy/imle_policy/datasets/lift.pkl'
if not os.path.exists(local_link):
    os.symlink(DRIVE_DATASET, local_link)
print('dataset ready at', local_link)

## 5. wandb login

In [ ]:
import wandb
wandb.login()  # paste your API key from wandb.ai/authorize when prompted

## 6. Point saved_weights at Drive

In [ ]:
os.makedirs(f'{DRIVE_ROOT}/saved_weights', exist_ok=True)
local_weights = '/content/imle_policy/imle_policy/saved_weights'
if os.path.islink(local_weights) or os.path.exists(local_weights):
    if os.path.islink(local_weights):
        os.remove(local_weights)
os.symlink(f'{DRIVE_ROOT}/saved_weights', local_weights)
print('saved_weights ->', os.path.realpath(local_weights))

## 7. Calibrate, then launch the full run

We don't have a measured per-epoch time for Lift on any GPU (all benchmarking in this project
so far was PushT-specific -- different `obs_dim`, `num_cameras=2` vs 1, `action_dim=7` vs 2).
Rather than guess, **run 10 epochs first** and read the actual `it/s` from the progress bar --
multiply by `num_epochs=500` from `configs/Lift_config.json` to get a real estimate before
committing GPU time to the full run. Ballpark expectation going in: probably same order of
magnitude as the PushT original-architecture Colab run (~5-6h for 500 epochs on an A100), likely
somewhat more given the extra camera and larger action space -- but treat that as a guess to be
replaced by this cell's actual measurement, not a plan.

In [ ]:
%cd /content/imle_policy/imle_policy

# Step 1: calibration -- interrupt this manually (Runtime -> Interrupt execution) once you've
# seen a stable it/s in the Batch progress bar (a few dozen steps is enough), then read the
# estimated total time before deciding whether to launch the full 500-epoch run below.
!python train.py --task Lift --method rs_imle \
    --wandb_run_name lift_calibration \
    --num_epochs 10

In [ ]:
# Step 2: the full run, once you've seen the calibration numbers above and are happy with the
# time estimate. Uses the task config's defaults (down_dims default from rs_imle_network.py,
# batch_size=64, num_epochs=500) -- edit flags here the same way as the PushT notebook if you
# want a specific down_dims/batch_size/loss variant instead.
!python train.py --task Lift --method rs_imle \
    --wandb_run_name lift_original_run

## 8. If the runtime disconnects
Re-run cells 1-2 and the dataset-symlink half of cell 4 (it'll skip the 25.8GB download since
`lift.pkl` is already on Drive) and the cell 6 symlink, then resume:
```python
!python train.py --resume_from saved_weights/lift_original_run_.../latest_checkpoint.pth
```
Same mechanism as every other run in this project -- model, optimizer, EMA, RNG state, and
wandb run id are all restored from the checkpoint.

**Can Colab keep running if you close your laptop?** Yes, as long as the *browser tab* (or the
Colab connection) stays open somewhere -- Colab's execution happens on Google's servers, not
your laptop, so closing the laptop's lid typically just puts the laptop to sleep and drops the
browser's connection to the runtime, which does NOT stop the remote execution by itself. The
practical risks are: (1) your OS may fully suspend networking on lid-close rather than just
idling, which would drop the websocket Colab uses to stream output/checkpoint the notebook UI
state (training itself keeps running server-side either way, but you won't see live output when
you reconnect until you re-open the tab); (2) Colab's own free-tier limits (~12h max session
length, and it disconnects idle runtimes after a period with no *browser* activity, not compute
activity -- so a long unattended run can still be cut short by Colab itself, independent of your
laptop). Colab Pro/Pro+ extends both limits. Either way, the `--resume_from` mechanism above
means a disconnect (from either cause) never loses more than the last lightweight-checkpoint
interval of progress.

## 9. Bring a checkpoint back to the local machine
Same as the PushT workflow -- Drive already has every checkpoint via the symlink in step 6.
See `colab/COLAB_WORKFLOW.md` for the local-eval instructions (`eval_colab_checkpoint.py`),
though note that one currently assumes PushT's `evaluate()`/env -- reusing it for Lift would
need pointing at `eval_policy_robomimic.py`'s `evaluate()` instead, which needs `robosuite`
installed locally too (not yet set up on the local machine, since local training here has been
PushT-only so far).